### Sanity Check Base CNN

In [ ]:
"""
Base CNN Model for 3D Pointing Detection
A simple sanity check model to establish baseline performance
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class BaseCNN(nn.Module):
    """
    Simple CNN for 3D pointing detection from RGB-D images.
    
    Input: 4-channel image (RGB + Depth) of shape (B, 4, H, W)
    Output: 
        - confidence: (B, 1) - probability of pointing gesture
        - vector: (B, 6) - two 3D points (wrist position + direction vector)
    """
    
    def __init__(self, input_height=720, input_width=1280, dropout_rate=0.3):
        super(BaseCNN, self).__init__()
        
        # Convolutional layers
        # Input: (B, 4, H, W)
        self.conv1 = nn.Conv2d(4, 32, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        
        # Calculate flattened size after convolutions
        # After 5 stride-2 convolutions: H/32 x W/32
        self.flat_size = 512 * (input_height // 32) * (input_width // 32)
        
        # Fully connected layers
        self.dropout = nn.Dropout(dropout_rate)
        
        # Shared features
        self.fc1 = nn.Linear(self.flat_size, 1024)
        self.fc2 = nn.Linear(1024, 512)
        
        # Classification head (pointing vs not pointing)
        self.fc_conf = nn.Linear(512, 1)
        
        # Regression head (6 values: 3 for wrist position, 3 for direction vector)
        self.fc_vec = nn.Linear(512, 6)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights using Kaiming initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input tensor of shape (B, 4, H, W)
            
        Returns:
            confidence: (B, 1) - sigmoid-activated confidence score
            vector: (B, 6) - predicted 3D vector components
        """
        # Convolutional feature extraction
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        
        # Output heads
        confidence = torch.sigmoid(self.fc_conf(x))
        vector = self.fc_vec(x)
        
        return confidence, vector

#### Test Model

In [ ]:
def test_base_cnn():
    """Test function to verify model architecture"""
    print("Testing BaseCNN...")
    
    # Create model
    model = BaseCNN(input_height=720, input_width=1280)
    
    # Create dummy input (batch_size=2, channels=4, height=720, width=1280)
    dummy_input = torch.randn(2, 4, 720, 1280)
    
    # Forward pass
    confidence, vector = model(dummy_input)
    
    print(f"Input shape: {dummy_input.shape}")
    print(f"Confidence output shape: {confidence.shape}")
    print(f"Vector output shape: {vector.shape}")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    assert confidence.shape == (2, 1), "Confidence output shape mismatch"
    assert vector.shape == (2, 6), "Vector output shape mismatch"
    
    print("✓ BaseCNN test passed!")


if __name__ == "__main__":
    test_base_cnn()

### YOLO-based CNN

In [ ]:
"""
YOLO-Inspired CNN Model for 3D Pointing Detection
Uses CSPDarknet backbone architecture for improved feature extraction
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    """Basic convolutional block with Conv, BatchNorm, and activation"""
    
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, activation='silu'):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        
        if activation == 'silu':
            self.act = nn.SiLU(inplace=True)
        elif activation == 'relu':
            self.act = nn.ReLU(inplace=True)
        elif activation == 'leaky':
            self.act = nn.LeakyReLU(0.1, inplace=True)
        else:
            self.act = nn.Identity()
    
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ResidualBlock(nn.Module):
    """Residual block used in Darknet"""
    
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        half_channels = channels // 2
        self.conv1 = ConvBlock(channels, half_channels, kernel_size=1, padding=0)
        self.conv2 = ConvBlock(half_channels, channels, kernel_size=3, padding=1)
    
    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.conv2(x)
        return x + residual


class CSPBlock(nn.Module):
    """Cross Stage Partial block from CSPNet"""
    
    def __init__(self, in_channels, out_channels, num_blocks=1, shortcut=True):
        super(CSPBlock, self).__init__()
        half_channels = out_channels // 2
        
        # Split
        self.conv1 = ConvBlock(in_channels, half_channels, kernel_size=1, padding=0)
        self.conv2 = ConvBlock(in_channels, half_channels, kernel_size=1, padding=0)
        
        # Residual blocks
        self.blocks = nn.Sequential(
            *[ResidualBlock(half_channels) for _ in range(num_blocks)]
        )
        
        # Merge
        self.conv3 = ConvBlock(half_channels, half_channels, kernel_size=1, padding=0)
        self.conv4 = ConvBlock(out_channels, out_channels, kernel_size=1, padding=0)
        
        self.shortcut = shortcut
    
    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x)
        
        x1 = self.blocks(x1)
        x1 = self.conv3(x1)
        
        x = torch.cat([x1, x2], dim=1)
        x = self.conv4(x)
        
        return x


class SPPBlock(nn.Module):
    """Spatial Pyramid Pooling block"""
    
    def __init__(self, in_channels, out_channels, kernel_sizes=[5, 9, 13]):
        super(SPPBlock, self).__init__()
        self.kernel_sizes = kernel_sizes
        self.conv1 = ConvBlock(in_channels, in_channels // 2, kernel_size=1, padding=0)
        self.conv2 = ConvBlock(in_channels // 2 * (len(kernel_sizes) + 1), out_channels, kernel_size=1, padding=0)
    
    def forward(self, x):
        x = self.conv1(x)
        features = [x]
        
        for kernel_size in self.kernel_sizes:
            padding = kernel_size // 2
            pooled = F.max_pool2d(x, kernel_size, stride=1, padding=padding)
            features.append(pooled)
        
        x = torch.cat(features, dim=1)
        x = self.conv2(x)
        return x


class YOLOBackbone(nn.Module):
    """
    YOLO-inspired backbone for 3D pointing detection
    Uses CSPDarknet architecture
    """
    
    def __init__(self, input_channels=4, base_channels=64, depth_multiple=1.0, width_multiple=1.0):
        super(YOLOBackbone, self).__init__()
        
        # Calculate channel sizes based on width_multiple
        def make_divisible(x, divisor=8):
            return int((x * width_multiple + divisor / 2) // divisor * divisor)
        
        # Stem
        self.stem = ConvBlock(input_channels, make_divisible(base_channels), kernel_size=6, stride=2, padding=2)
        
        # Stage 1
        self.stage1 = nn.Sequential(
            ConvBlock(make_divisible(base_channels), make_divisible(base_channels * 2), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 2), make_divisible(base_channels * 2), num_blocks=max(round(3 * depth_multiple), 1))
        )
        
        # Stage 2
        self.stage2 = nn.Sequential(
            ConvBlock(make_divisible(base_channels * 2), make_divisible(base_channels * 4), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 4), make_divisible(base_channels * 4), num_blocks=max(round(6 * depth_multiple), 1))
        )
        
        # Stage 3
        self.stage3 = nn.Sequential(
            ConvBlock(make_divisible(base_channels * 4), make_divisible(base_channels * 8), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 8), make_divisible(base_channels * 8), num_blocks=max(round(9 * depth_multiple), 1))
        )
        
        # Stage 4
        self.stage4 = nn.Sequential(
            ConvBlock(make_divisible(base_channels * 8), make_divisible(base_channels * 16), kernel_size=3, stride=2, padding=1),
            CSPBlock(make_divisible(base_channels * 16), make_divisible(base_channels * 16), num_blocks=max(round(3 * depth_multiple), 1))
        )
        
        # SPP
        self.spp = SPPBlock(make_divisible(base_channels * 16), make_divisible(base_channels * 16))
        
        self.final_channels = make_divisible(base_channels * 16)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.spp(x)
        return x


class YOLOPointingNet(nn.Module):
    """
    Complete YOLO-based network for 3D pointing detection
    
    Input: 4-channel image (RGB + Depth) of shape (B, 4, H, W)
    Output: 
        - confidence: (B, 1) - probability of pointing gesture
        - vector: (B, 6) - two 3D points (wrist position + direction vector)
    """
    
    def __init__(self, input_height=720, input_width=1280, input_channels=4, 
                 depth_multiple=0.33, width_multiple=0.5, dropout_rate=0.2):
        """
        Args:
            input_height: Input image height
            input_width: Input image width
            input_channels: Number of input channels (4 for RGB-D)
            depth_multiple: Depth scaling factor (0.33 for small, 0.67 for medium, 1.0 for large)
            width_multiple: Width scaling factor (0.5 for small, 0.75 for medium, 1.0 for large)
            dropout_rate: Dropout probability
        """
        super(YOLOPointingNet, self).__init__()
        
        # Backbone
        self.backbone = YOLOBackbone(input_channels, base_channels=64, 
                                     depth_multiple=depth_multiple, 
                                     width_multiple=width_multiple)
        
        # Calculate flattened size
        # After 5 stride-2 operations: H/32 x W/32
        self.flat_size = self.backbone.final_channels * (input_height // 32) * (input_width // 32)
        
        # Global Average Pooling (alternative to flattening)
        self.use_gap = True
        
        if self.use_gap:
            self.gap = nn.AdaptiveAvgPool2d(1)
            fc_input_size = self.backbone.final_channels
        else:
            fc_input_size = self.flat_size
        
        # Head network
        self.dropout = nn.Dropout(dropout_rate)
        
        # Shared feature layers
        self.fc1 = nn.Linear(fc_input_size, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        
        # Classification head (pointing vs not pointing)
        self.fc_conf = nn.Linear(256, 1)
        
        # Regression head (6 values: 3 for wrist position, 3 for direction vector)
        self.fc_vec1 = nn.Linear(256, 128)
        self.fc_vec2 = nn.Linear(128, 6)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input tensor of shape (B, 4, H, W)
            
        Returns:
            confidence: (B, 1) - sigmoid-activated confidence score
            vector: (B, 6) - predicted 3D vector components
        """
        # Backbone feature extraction
        x = self.backbone(x)
        
        # Pooling and flattening
        if self.use_gap:
            x = self.gap(x)
            x = x.view(x.size(0), -1)
        else:
            x = x.view(x.size(0), -1)
        
        # Shared features
        x = F.silu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.silu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        
        # Classification head
        confidence = torch.sigmoid(self.fc_conf(x))
        
        # Regression head
        vector = F.silu(self.fc_vec1(x))
        vector = self.fc_vec2(vector)
        
        return confidence, vector


#### Test model

In [ ]:
def create_yolo_pointing_net(model_size='small', input_height=720, input_width=1280):
    """
    Factory function to create YOLOPointingNet with different sizes
    
    Args:
        model_size: One of 'small', 'medium', 'large'
        input_height: Input image height
        input_width: Input image width
    
    Returns:
        YOLOPointingNet model
    """
    size_configs = {
        'small': {'depth_multiple': 0.33, 'width_multiple': 0.5},
        'medium': {'depth_multiple': 0.67, 'width_multiple': 0.75},
        'large': {'depth_multiple': 1.0, 'width_multiple': 1.0}
    }
    
    if model_size not in size_configs:
        raise ValueError(f"model_size must be one of {list(size_configs.keys())}")
    
    config = size_configs[model_size]
    
    return YOLOPointingNet(
        input_height=input_height,
        input_width=input_width,
        depth_multiple=config['depth_multiple'],
        width_multiple=config['width_multiple']
    )


def test_yolo_pointing_net():
    """Test function to verify model architecture"""
    print("Testing YOLOPointingNet...")
    
    for size in ['small', 'medium', 'large']:
        print(f"\n{size.upper()} model:")
        model = create_yolo_pointing_net(model_size=size, input_height=720, input_width=1280)
        
        # Create dummy input
        dummy_input = torch.randn(2, 4, 720, 1280)
        
        # Forward pass
        with torch.no_grad():
            confidence, vector = model(dummy_input)
        
        print(f"  Input shape: {dummy_input.shape}")
        print(f"  Confidence output shape: {confidence.shape}")
        print(f"  Vector output shape: {vector.shape}")
        print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
        
        assert confidence.shape == (2, 1), f"Confidence output shape mismatch for {size}"
        assert vector.shape == (2, 6), f"Vector output shape mismatch for {size}"
    
    print("\n✓ All YOLOPointingNet tests passed!")


if __name__ == "__main__":
    test_yolo_pointing_net()